# PySpark with MinIO: Reading, Writing, and Basic EDA

This notebook demonstrates how to use PySpark to:
1. Read a local CSV file.
2. Write the data to a MinIO S3 bucket.
3. Read the data back from MinIO.
4. Perform some basic Exploratory Data Analysis (EDA).

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, dayofweek, hour, avg, count, desc

# MinIO Connection Details
minio_endpoint = "http://minio:9000"
minio_access_key = "minioadmin"
minio_secret_key = "minioadmin"
bucket_name = "tripdata"

## 1. Initialize Spark Session

We need to configure the Spark session to connect to our MinIO server. This involves setting the S3A endpoint, access keys, and other necessary Hadoop configurations.

In [2]:
spark = (
    SparkSession.builder.appName("PySparkMinIOEDA")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print("Spark Session Created Successfully")

Spark Session Created Successfully


## 2. Create the MinIO Bucket

Before we can write data, we need to ensure the bucket exists. We'll use the `boto3` library for this.

In [9]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=minio_endpoint,
    aws_access_key_id=minio_access_key,
    aws_secret_access_key=minio_secret_key,
    config=Config(signature_version='s3v4')
)

try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Bucket '{bucket_name}' created successfully.")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{bucket_name}' already exists.")
except Exception as e:
    print(f"An error occurred: {e}")

Bucket 'tripdata' already exists.


## 3. Read Local CSV File

We'll read the `BEAD-Rebu_TripData.csv` file from the mounted `data` directory into a Spark DataFrame.

In [10]:
local_csv_path = "/home/jovyan/data/BEAD_Rebu_TripData.csv"

df_local = spark.read.csv(local_csv_path, header=True, inferSchema=True)

print("CSV file read into DataFrame. Schema:")
df_local.printSchema()

print("\nSample data:")
df_local.show(5)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/data/BEAD-Rebu_TripData.csv.

## 4. Write DataFrame to MinIO

Now, we'll write the DataFrame to our MinIO bucket. We'll save it in the efficient Parquet format.

In [ ]:
minio_path = f"s3a://{bucket_name}/BEAD-Rebu_TripData.parquet"

df_local.write.mode("overwrite").parquet(minio_path)

print(f"DataFrame successfully written to {minio_path}")

## 5. Read Data Back from MinIO

To confirm that the data was written correctly, we will now read it back from MinIO into a new DataFrame.

In [ ]:
df_minio = spark.read.parquet(minio_path)

print("Data read back from MinIO. Schema:")
df_minio.printSchema()

print("\nSample data from MinIO:")
df_minio.show(5)

## 6. Basic Exploratory Data Analysis (EDA)

Now that we have our data in a Spark DataFrame read from MinIO, let's perform some simple EDA.

In [ ]:
# --- Data Cleaning and Transformation ---
# Convert the 'Date' column from string to date type
# The date format in the CSV is 'd-MMM-yy'
df_transformed = df_minio.withColumn("TripDate", to_date(col("Date"), 'd-MMM-yy'))

# --- EDA Questions ---

# 1. What is the total number of trips?
total_trips = df_transformed.count()
print(f"Total number of trips: {total_trips}")

# 2. What is the average trip distance?
avg_distance = df_transformed.select(avg("Distance Travelled")).first()[0]
print(f"Average trip distance: {avg_distance:.2f} km")

# 3. What are the top 5 most popular pickup districts?
print("\nTop 5 most popular pickup districts:")
popular_pickups = df_transformed.groupBy("Pickup District").agg(count("*").alias("trip_count")) \
                                .orderBy(desc("trip_count"))
popular_pickups.show(5)

# 4. How many trips occur on each day of the week?
print("\nNumber of trips per day of the week:")
trips_by_day = df_transformed.withColumn("DayOfWeek", dayofweek(col("TripDate"))) \
                             .groupBy("DayOfWeek").agg(count("*").alias("trip_count")) \
                             .orderBy("DayOfWeek") # 1=Sun, 2=Mon, ..., 7=Sat
trips_by_day.show()

# 5. What is the busiest hour of the day for trips?
print("\nBusiest hour of the day:")
trips_by_hour = df_transformed.groupBy("Hour of Day").agg(count("*").alias("trip_count")) \
                               .orderBy(desc("trip_count"))
trips_by_hour.show(1)

## 7. Stop the Spark Session

It's good practice to stop the Spark session when you're done.

In [ ]:
spark.stop()